# Classifier Evaluation Results Analysis

This read-only notebook analyzes one saved classifier evaluation run. It compares feature sets, parameter choices, scorer LLMs, evaluation datasets, `Mixed` versus `Mixed+BBC` training, and SemEval localization. All claims are descriptive and depend on the coverage available in the selected run.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from notebooks.utils.classification_results_analysis_utils import (
    aggregate_by_dataset, aggregate_by_llm, available_table, bbc_safety_metrics,
    configuration_ranking, document_metrics, load_classification_run,
    mixed_bbc_comparison, plot_aggregate_bars, plot_feature_set_heatmap,
    plot_regime_deltas, plot_semeval_ranking, plot_window_heatmap,
    semeval_boundary_ranking, semeval_exclusions, semeval_region_ranking,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
RUN_ID = "feature-vector-classification-2026-09-15-rerun-2"
results = load_classification_run(DATA_ROOT, run_id=RUN_ID)
metrics = document_metrics(results)
pd.set_option("display.max_columns", 50)

## Data Quality And Experiment Inventory

The provenance table identifies whether each export was read from its current headered schema or reconstructed from a known legacy positional schema. A legacy warning means the original result files should be regenerated before publication.

In [ ]:
display(results.provenance)
if results.provenance["source"].eq("legacy positional recovery").any():
    display(Markdown("**Warning:** one or more exports use legacy positional recovery. Regenerate this run with the corrected streaming CSV writer before treating it as a publication artifact."))

run_summary = available_table(results, "run_summary")
coverage = available_table(results, "evaluation_coverage")
display(run_summary.groupby(["stage", "status"], dropna=False).size().rename("rows").reset_index() if not run_summary.empty else pd.DataFrame())
display(coverage.groupby(["dataset", "partition", "availability_status"], dropna=False)["row_count"].sum().reset_index() if not coverage.empty else pd.DataFrame())
display(metrics[[column for column in ("scorer_llm", "model_family", "feature_set", "training_regime", "training_window", "evaluation_window", "dataset") if column in metrics]].drop_duplicates().sort_values(list(metrics.columns.intersection(["scorer_llm", "feature_set", "training_window", "evaluation_window", "dataset"]))))

## Document Classification

The primary ranking uses test macro-F1 on binary Kaggle, MAGE, and Outfox evaluation rows. BBC is HG-only and is evaluated later through false-positive safety metrics rather than binary macro-F1.

In [ ]:
ranking = configuration_ranking(metrics)
llm_summary = aggregate_by_llm(metrics)
dataset_summary = aggregate_by_dataset(metrics)
display(ranking.head(30))
display(llm_summary)
display(dataset_summary)

plot_aggregate_bars(llm_summary, category="scorer_llm", title=f"{RUN_ID}: mean binary test macro-F1 by scorer LLM")
plt.show()
plot_aggregate_bars(dataset_summary, category="dataset", title=f"{RUN_ID}: mean test macro-F1 by evaluation dataset")
plt.show()

## Feature Sets And Classification Parameters

Heatmaps average available binary test rows. Inspect the ranked table above before selecting a configuration: an average can conceal source-specific weaknesses. If only one classifier family is present, this run cannot compare estimator types.

In [ ]:
for dataset, dataset_metrics in metrics.groupby("dataset", sort=True):
    display(Markdown(f"### {dataset}"))
    display(dataset_metrics.groupby(["scorer_llm", "feature_set", "training_window", "evaluation_window"], dropna=False)[["macro_f1", "roc_auc", "pr_auc", "balanced_accuracy", "false_positive_rate"]].mean().sort_values("macro_f1", ascending=False).head(20))
    plot_feature_set_heatmap(dataset_metrics, title=f"{RUN_ID}: feature-set macro-F1 on {dataset}")
    plt.show()
    plot_window_heatmap(dataset_metrics, title=f"{RUN_ID}: window macro-F1 on {dataset}")
    plt.show()

model_families = sorted(metrics["model_family"].dropna().astype(str).unique())
display(pd.DataFrame({"model_family": model_families}))
if len(model_families) < 2:
    display(Markdown("Only one classifier family is available in this run, so these results compare representations and parameters rather than estimator types."))

## Mixed Versus Mixed+BBC Training

Matching holds scorer LLM, classifier family, feature set, training window, evaluation window, evaluation dataset, and partition constant. The delta is `Mixed+BBC - Mixed` on binary-source macro-F1. BBC results are shown separately as HG false-positive safety outcomes.

In [ ]:
regime_comparison = mixed_bbc_comparison(metrics)
bbc_safety = bbc_safety_metrics(results)
if regime_comparison.empty:
    display(Markdown("No matched `Mixed` and `Mixed+BBC` binary configurations were available in this run."))
else:
    display(regime_comparison.sort_values("macro_f1_delta", ascending=False))
    plot_regime_deltas(regime_comparison, title=f"{RUN_ID}: binary macro-F1 effect of adding BBC")
    plt.show()

if bbc_safety.empty:
    display(Markdown("No BBC test rows were available for HG false-positive analysis."))
else:
    display(bbc_safety[[column for column in ("scorer_llm", "feature_set", "training_regime", "training_window", "evaluation_window", "false_positive_rate", "predicted_mg_rate", "hg_support") if column in bbc_safety]])

## SemEval Region And Boundary Localization

MG-region F1 and boundary error answer different questions. Region ranking is descending by MG F1; boundary ranking is ascending by mean absolute token error. Exclusions reveal unavailable texts and configurations.

In [ ]:
region_ranking = semeval_region_ranking(results)
boundary_ranking = semeval_boundary_ranking(results)
exclusions = semeval_exclusions(results)
display(region_ranking.head(30))
display(boundary_ranking.head(30))
display(exclusions.groupby("reason", dropna=False).size().rename("texts").sort_values(ascending=False).head(20).reset_index() if not exclusions.empty else pd.DataFrame())
if not region_ranking.empty:
    plot_semeval_ranking(region_ranking, metric="mg_f1", title=f"{RUN_ID}: SemEval MG-region F1")
    plt.show()
if not boundary_ranking.empty:
    plot_semeval_ranking(boundary_ranking, metric="mean_absolute_token_error", title=f"{RUN_ID}: SemEval boundary error", ascending=True)
    plt.show()

## Research Conclusions

The cell below generates descriptive conclusions from the displayed tables. It does not test statistical significance and explicitly reports missing matched comparisons.

In [ ]:
conclusions = [f"## Conclusions for `{RUN_ID}`"]
if not ranking.empty:
    best = ranking.iloc[0]
    conclusions.append(f"- The strongest observed binary-source configuration is `{best['scorer_llm']}` with `{best['feature_set']}` features, trained under `{best['training_regime']}` at {best['training_window']} and evaluated at {best['evaluation_window']} tokens (mean macro-F1 {best['mean']:.3f}; worst source {best['minimum']:.3f}).")
if not dataset_summary.empty:
    hardest = dataset_summary.sort_values("mean").iloc[0]
    conclusions.append(f"- `{hardest['dataset']}` is the most difficult available binary source by mean macro-F1 ({hardest['mean']:.3f}).")
if regime_comparison.empty:
    conclusions.append("- No matched `Mixed` and `Mixed+BBC` binary configurations are available, so this run cannot isolate the effect of adding BBC training data.")
else:
    conclusions.append(f"- Adding BBC changes matched binary macro-F1 by a mean of {regime_comparison['macro_f1_delta'].mean():+.3f}; inspect BBC false-positive rates separately before selecting a regime.")
if not region_ranking.empty:
    region_best = region_ranking.iloc[0]
    conclusions.append(f"- The leading SemEval region configuration has MG-region F1 {region_best['mg_f1']:.3f}.")
if not boundary_ranking.empty:
    boundary_best = boundary_ranking.iloc[0]
    conclusions.append(f"- The best available SemEval boundary configuration has mean absolute token error {boundary_best['mean_absolute_token_error']:.2f}; it may differ from the region-F1 winner.")
if results.provenance['source'].eq("legacy positional recovery").any():
    conclusions.append("- These values were reconstructed from legacy exports. Regenerate the run with the corrected CSV writer before publication.")
conclusions.append("- Results are descriptive only: this run may have incomplete coverage, uses one cohort seed, and may contain only one classifier family.")
display(Markdown("\n".join(conclusions)))

## Universal Cross-Source Transfer

Universal candidates are ranked only on binary-source test macro-F1. Single-source models exclude their in-distribution source; `Mixed` and `Mixed+BBC` include every binary source.

In [ ]:
from notebooks.utils.classification_results_analysis_utils import (
    best_universal_by_llm, pareto_universal_configurations, plot_transfer_asymmetry,
    plot_transfer_heatmap, plot_universal_frontier, plot_stability_profiles,
    plot_semeval_tradeoff, transfer_matrix, transfer_scores, universal_ranking,
)

transfer = transfer_scores(metrics)
universal = universal_ranking(metrics)
best_by_llm = best_universal_by_llm(metrics)
pareto = pareto_universal_configurations(universal)
display(universal)
display(best_by_llm)
display(pareto)

available_regimes = sorted(metrics["training_regime"].dropna().unique())
if len(available_regimes) <= 1:
    display(Markdown(f"**Availability warning:** this run contains only {len(available_regimes)} training regime ({', '.join(available_regimes) or 'none'}), so cross-regime comparisons are unavailable."))

configuration_columns = ["scorer_llm", "model_family", "feature_set", "training_regime", "training_window", "evaluation_window"]
if not universal.empty:
    for _, configuration in universal.iterrows():
        selected = {column: configuration[column] for column in configuration_columns}
        matrix = transfer_matrix(metrics, selected)
        if not matrix.empty:
            display(Markdown(f"### {configuration['scorer_llm']} | {configuration['feature_set']} | {configuration['training_regime']}"))
            plot_transfer_heatmap(matrix, title=f"{RUN_ID}: transfer matrix")
            plt.show()
if not best_by_llm.empty:
    for _, configuration in best_by_llm.iterrows():
        selected = {column: configuration[column] for column in configuration_columns}
        matrix = transfer_matrix(metrics, selected)
        if not matrix.empty:
            display(Markdown(f"### Best universal candidate: {configuration['scorer_llm']}"))
            plot_transfer_heatmap(matrix, title=f"{RUN_ID}: best {configuration['scorer_llm']} transfer matrix")
            plt.show()

## Robust Universal Candidates

The frontier highlights configurations that are not jointly dominated on worst eligible source macro-F1 and mean eligible source macro-F1.

In [ ]:
if not universal.empty:
    plot_universal_frontier(universal, title=f"{RUN_ID}: universal robustness frontier")
    plt.show()
if not transfer.empty and not best_by_llm.empty:
    plot_stability_profiles(transfer, best_by_llm, title=f"{RUN_ID}: best-universal stability profiles")
    plt.show()

## Transfer Asymmetry

Directional diagrams show how each best-per-LLM configuration transfers among the three binary sources; missing cells remain unavailable rather than being treated as zero.

In [ ]:
if not best_by_llm.empty:
    for _, configuration in best_by_llm.iterrows():
        selected = {column: configuration[column] for column in configuration_columns}
        matrix = transfer_matrix(metrics, selected)
        if not matrix.empty:
            plot_transfer_asymmetry(matrix, title=f"{RUN_ID}: {configuration['scorer_llm']} transfer asymmetry")
            plt.show()

## SemEval Trade-off

The SemEval plot compares MG-region F1 with boundary localization error only for configurations represented in both summaries. BBC HG-only false-positive safety remains separate from binary transfer macro-F1.

In [ ]:
if not region_ranking.empty and not boundary_ranking.empty:
    plot_semeval_tradeoff(region_ranking, boundary_ranking, title=f"{RUN_ID}: SemEval MG-region F1 versus boundary error")
    plt.show()

transfer_conclusions = [
    "### Transfer analysis conclusions",
    "- Universal candidates are selected first by worst eligible binary-source test macro-F1, then by mean eligible macro-F1.",
]
expected_regimes = ["Kaggle", "MAGE", "Outfox", "Mixed", "Mixed+BBC"]
missing_regimes = [regime for regime in expected_regimes if regime not in available_regimes]
transfer_conclusions.append(f"- Training regimes missing from this run: {', '.join(missing_regimes) if missing_regimes else 'none'}.")
if not universal.empty:
    best_universal = universal.iloc[0]
    transfer_conclusions.append(f"- The current top universal candidate is `{best_universal['scorer_llm']}` with worst eligible macro-F1 {best_universal['worst_source_macro_f1']:.3f} and mean eligible macro-F1 {best_universal['mean_source_macro_f1']:.3f}.")
display(Markdown("\n".join(transfer_conclusions)))